# agentic_alpha_session_effect_search_v1

Diagnostic R-G-E-V-D card for overnight/intraday session effects. It maps external overnight premium, tug-of-war, and overnight momentum references into qlib-native SOXL/SOXX OHLC features, then closes the branch against the production `qlib_paper_sim_v1` return and MaxDD gate.

## R-G-E-V-D Route

- Research/Retrieve: agentic-alpha workflow, overnight risk premium, tug-of-war, overnight momentum, and ETF trend/gap examples.
- Generate: close-to-next-open overnight momentum, open-to-close gap reversal, and close-to-close tug-of-war variants.
- Evaluate/Validate: compare each grid row with the production baseline over `2019-01-02` to `2026-06-09`.
- Distill: promote only if return beats the baseline and MaxDD is no worse than `-20%`.

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.factor import build_session_effect_frame, compact_session_effect_summary, evaluate_session_effect_candidates
from src.factor.session_effect import SESSION_EFFECT_SOURCE_FIELDS
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_session_effect_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
START = '2019-01-02'
END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
TRANSACTION_COST = 0.0002

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  train_start: 2010-03-11
  test:
    start: {START}
    end: {END}
rgevd:
  branch: session_effect_overnight_intraday
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use Research/Retrieve/Generate/Evaluate/Validate/Distill loops and close every generated branch with evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'github',
            'title': 'llm-quant overnight momentum strategy family',
            'url': 'https://github.com/45ck/llm-quant',
            'extracted_signal': 'Overnight Momentum treats rolling overnight gaps as a candidate predictor and requires crash filters for leveraged ETFs.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'A Tug of War: Overnight Versus Intraday Expected Returns',
            'url': 'https://personal.lse.ac.uk/polk/research/TugOfWar.pdf',
            'extracted_signal': 'Separate overnight and intraday return components; smoothed overnight-minus-intraday spreads may forecast future returns.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Overnight Return Momentum and the Timing of Trading Volume',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5004991',
            'extracted_signal': 'Overnight return momentum links prior overnight winners and intraday trading patterns to subsequent overnight returns.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Does Overnight News Explain Overnight Returns?',
            'url': 'https://arxiv.org/html/2507.04481v1',
            'extracted_signal': 'Overnight gains can dominate market returns; continuation and reversal patterns differ across overnight and intraday sessions.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'Concretum Research overnight risk premium discussion',
            'url': 'https://x.com/ConcretumR',
            'extracted_signal': 'Market practitioners revisit overnight risk premium as a distinct, tradable session effect.',
            'retrieval_status': 'search_snippet_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
print(f'baseline_return={baseline.return_pct:.4f}, baseline_maxdd={baseline.max_drawdown_pct:.4f}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


baseline_return=6570.0233, baseline_maxdd=-85.2317


In [2]:
qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
raw = D.features(
    ['SOXL', 'SOXX'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time='2010-03-11',
    end_time='2026-06-10',
    freq='day',
)
features = build_session_effect_frame(raw, start=START, end=END, min_history=20)
metrics = evaluate_session_effect_candidates(
    features,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    cost_per_turnover=TRANSACTION_COST,
)
branches = ['intraday_gap_reversal', 'overnight_momentum', 'tug_of_war_close_to_close']
search_summary = compact_session_effect_summary(
    metrics,
    branches=branches,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
search_summary['baseline'] = {
    'name': baseline.name,
    'return_pct': float(baseline.return_pct),
    'max_drawdown_pct': float(baseline.max_drawdown_pct),
    'benchmark_return_pct': float(baseline.benchmark_return_pct),
    'recorder_id': baseline.recorder_id,
}
search_summary['external_source_counts'] = external_sources['source_type'].value_counts().sort_index().to_dict()
search_summary['feature_evidence'] = {
    'feature_rows': int(len(features)),
    'metric_rows': int(len(metrics)),
    'source_coverage_start': features.attrs.get('source_coverage_start'),
    'source_coverage_end': features.attrs.get('source_coverage_end'),
    'test_start': START,
    'test_end': END,
    'sessions': ['close_to_next_open', 'open_to_close', 'close_to_next_close'],
}

candidate_rows = []
for branch in branches:
    rows = metrics.loc[metrics['branch'] == branch].copy()
    best_return = rows.sort_values('return_pct', ascending=False).iloc[0]
    drawdown_rows = rows.loc[rows['passes_drawdown_gate'] == True].sort_values('return_pct', ascending=False)  # noqa: E712
    for label, row in [('best_return', best_return), ('best_drawdown_gate', drawdown_rows.iloc[0] if not drawdown_rows.empty else None)]:
        if row is None:
            continue
        candidate_rows.append(
            {
                'branch': branch,
                'candidate_type': label,
                'candidate': row['candidate'],
                'session': row['session'],
                'return_pct': float(row['return_pct']),
                'baseline_return_pct': float(row['baseline_return_pct']),
                'return_improvement_pct': float(row['return_improvement_pct']),
                'max_drawdown_pct': float(row['max_drawdown_pct']),
                'max_drawdown_floor': float(row['max_drawdown_floor']),
                'turnover': float(row['turnover']),
                'max_exposure': float(row['max_exposure']),
                'trade_count': int(row['trade_count']),
                'beats_baseline_return': bool(row['beats_baseline_return']),
                'passes_drawdown_gate': bool(row['passes_drawdown_gate']),
                'decision': str(row['decision']),
                'decision_basis': str(row['decision_basis']),
                'params': row['params'],
                'grid_rows': int(len(rows)),
                'coverage_start': row['coverage_start'],
                'coverage_end': row['coverage_end'],
            }
        )
candidate_metrics = pd.DataFrame(candidate_rows)
candidate_metrics.to_csv(CARD / 'agentic_alpha_session_effect_candidate_metrics.csv', index=False)
(CARD / 'agentic_alpha_session_effect_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)

lines = ['# Failure Modes', '']
for branch, result in search_summary['branch_results'].items():
    best_dd = result['best_drawdown_gate_return_pct']
    best_dd_text = 'none' if best_dd is None else f"{best_dd:.2f}% at MaxDD {result['best_drawdown_gate_max_drawdown_pct']:.2f}%"
    lines.append(
        f"- {branch}: passes_both_count={result['passes_both_count']}; best_return={result['best_return_pct']:.2f}% "
        f"with MaxDD={result['best_return_max_drawdown_pct']:.2f}%; best_drawdown_gate={best_dd_text}."
    )
lines.append('- next_action: do not repeat pure overnight/intraday OHLC session splitting unless a new independent signal source is added.')
(CARD / 'failure_modes.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(json.dumps({
    'passes_both_count': search_summary['passes_both_count'],
    'branch_results': search_summary['branch_results'],
}, indent=2, sort_keys=True))

[56601:MainThread](2026-06-23 19:06:40,921) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56601:MainThread](2026-06-23 19:06:40,923) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56601:MainThread](2026-06-23 19:06:40,923) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{
  "branch_results": {
    "intraday_gap_reversal": {
      "best_drawdown_gate_available": false,
      "best_drawdown_gate_candidate": null,
      "best_drawdown_gate_max_drawdown_pct": null,
      "best_drawdown_gate_return_pct": null,
      "best_drawdown_gate_trade_count": null,
      "best_return_candidate": "intraday_gap_reversal_open_to_close_scale1_14100",
      "best_return_max_drawdown_pct": -73.87873554863836,
      "best_return_pct": 291.75756530149624,
      "best_return_trade_count": 1487,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 1728,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "overnight_momentum": {
      "best_drawdown_gate_available": false,
      "best_drawdown_gate_candidate": null,
      "best_drawdown_gate_max_drawdown_pct": null,
      "best_drawdown_gate_return_pct": null,
      "best_drawdown_gate_trade_count": null,
      "be

In [3]:
R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        source_url='https://github.com/JacobDu/agentic-alpha',
        metric_rows=len(metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_session_effect_search_summary.json').write_text(
        json.dumps(search_summary, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    for path in [
        CARD / 'agentic_alpha_session_effect_candidate_metrics.csv',
        CARD / 'agentic_alpha_session_effect_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_session_effect')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
search_summary['recorder_id'] = recorder_id
search_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
(CARD / 'agentic_alpha_session_effect_search_summary.json').write_text(
    json.dumps(search_summary, indent=2, sort_keys=True),
    encoding='utf-8',
)
print(f'recorder_id={recorder_id}')

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[56601:MainThread](2026-06-23 19:06:42,655) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_session_effect_search_v1.


[56601:MainThread](2026-06-23 19:06:42,657) INFO - qlib.workflow - [exp.py:258] - Experiment 120435610749988215 starts running ...


[56601:MainThread](2026-06-23 19:06:42,715) INFO - qlib.workflow - [recorder.py:345] - Recorder 2f9a0f1f45ce42fba25756854c00026c starts running under Experiment 120435610749988215 ...


[56601:MainThread](2026-06-23 19:06:42,777) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


recorder_id=2f9a0f1f45ce42fba25756854c00026c


In [4]:
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted',
    'as_of': END,
    'data_range': [START, END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'require_return_above_baseline': True,
    },
    'search_report': search_summary,
    'branch_results': search_summary['branch_results'],
    'external_source_counts': search_summary['external_source_counts'],
    'session_effect': {
        'selected_features': SESSION_EFFECT_SOURCE_FIELDS,
        'sessions': ['close_to_next_open', 'open_to_close', 'close_to_next_close'],
    },
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_session_effect_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_session_effect_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_session_effect_candidate_metrics.csv',
            'agentic_alpha_session_effect_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')

branch_results = search_summary['branch_results']
best_branch = max(branch_results, key=lambda name: branch_results[name]['best_return_pct'])
best = branch_results[best_branch]
conclusion = f'''# 结论

- what_ran: 执行 {EXPERIMENT}，用 qlib `D.features` 读取 SOXL/SOXX OHLC/VIX，测试 overnight momentum、intraday gap reversal、tug-of-war close-to-close 三条 session-effect 分支。
- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; passes_both_count={search_summary['passes_both_count']}。
- regime_window_difference: 最强收益分支是 {best_branch}，best_return={best['best_return_pct']:.2f}%, MaxDD={best['best_return_max_drawdown_pct']:.2f}%；回撤门控版本仍远低于生产基准。
- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。
- risks: overnight/intraday 分解解释了部分路径差异，但单独使用 OHLC session effect 无法压住 SOXL 杠杆路径回撤；高收益版本仍遭遇深度回撤。
- next_action: 需要新的独立信息源，例如期权曲面、订单流、或更长 intraday demand/supply imbalance；不要重复纯 session split 搜索。
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(json.dumps({'summary': 'written', 'promotion_decision': summary['promotion_decision']}, indent=2))

{
  "summary": "written",
  "promotion_decision": "not_promoted"
}
